# カメラキャリブレーション

このノートブックは、CSIカメラからのキャプチャー画像を表示し、カメラの動作確認やキャリブレーションを行うためのものです。

## Jetsonボード情報の取得

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO
    BOARD_NAME = "JETSON_ORIN_NANO"

# ---------- 2. ボード別定義 ----------
product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

product_name = product_names.get(BOARD_NAME, "未知のボード")
print("------------------------------------------------------------")
print(f"検出されたボード: {product_name}")
print("------------------------------------------------------------")

## カメラの初期化

In [ ]:
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
import ipywidgets
import traitlets
from IPython.display import display

## カメラ確認

In [ ]:
# カメラデバイスインデックスを選択するウィジェット
camera_devices_widget = ipywidgets.Dropdown(options=[], description="カメラデバイス")

# カメラ画像を表示するウィジェット
camera_widget = ipywidgets.Image(format='jpeg', width=224, height=224)

# カメラ画像へのグリッド描画有無の選択ウィジェット
draw_grids_checkbox = ipywidgets.Checkbox(description="グリッド描画", value=True)

# スナップショットを表示するウィジェット
snapshot_widget = ipywidgets.Image(format='jpeg', width=224, height=224)

# スナップショットボタン
snapshot_button = ipywidgets.Button(description='スナップショット')

# スナップショット保存ボタン
save_button = ipywidgets.Button(description='画像を保存')

# ファイル名入力ウィジェット
filename_widget = ipywidgets.Text(
    value='snapshot.jpg',
    description='ファイル名:',
    style={'description_width': 'initial'}
)

# カメラ停止ボタン
stop_camera_button = ipywidgets.Button(description='カメラ停止')

# ステータスメッセージ
status_widget = ipywidgets.Label(value='')

In [ ]:
import cv2
import glob

CAMERA_DEVICE_PREFIX = "/dev/video"

# カメラリソース
camera_device_index = None
camera = None
camera_link = None

# 現在のスナップショットを保持
current_snapshot = None

#
do_draw_grids = True

def start_camera(change):
    global camera_device_index, camera, camera_link

    if camera is not None:
        stop_camera()
    
    # カメラデバイスインデックスを更新
    camera_device_index = change["new"]
    
    # カメラの初期化（デバイス0、224x224、30fps）
    camera = CSICamera(capture_device=camera_device_index, width=224, height=224, capture_fps=30)

    # カメラを起動
    camera.running = True
    
    # カメラの画像をウィジェットに連携
    camera_link = traitlets.dlink((camera, 'value'), (camera_widget, 'value'), transform=draw_grids_and_convert_to_jpeg)

    status_widget.value = f"カメラ[{camera_device_index}]を起動しました"
    
def take_snapshot(button):
    global current_snapshot
    # 現在のカメラ画像をコピー
    current_snapshot = camera.value.copy()
    # スナップショットウィジェットに表示
    snapshot_widget.value = bgr8_to_jpeg(current_snapshot)
    status_widget.value = 'スナップショットを取得しました'

def save_snapshot(button):
    global current_snapshot
    if current_snapshot is not None:
        filepath = f'./calibration/{filename_widget.value}'
        # ディレクトリが存在しない場合は作成
        os.makedirs('./calibration', exist_ok=True)
        # 画像を保存
        cv2.imwrite(filepath, current_snapshot)
        status_widget.value = f'画像を保存しました: {filepath}'
    else:
        status_widget.value = '先にスナップショットを取得してください'

def draw_grids_and_convert_to_jpeg(img):
    from fabo.annotation import draw_grids

    if draw_grids_checkbox.value:
        img = draw_grids(img)
    return bgr8_to_jpeg(img)

def stop_camera(button=None):
    global camera_device_index, camera, camera_link
    if camera is not None and camera.running:
        # カメラを停止
        camera.running = False
        camera.cap.release()
        
        # camera_device_index は意図的に None に初期化しない
        camera = None
        camera_link = None

        status_widget.value = f"カメラ[{camera_device_index}]を停止しました"
    elif camera_device_index is not None:
        status_widget.value = f"カメラ[{camera_device_index}]は既に停止しています"
    else:
        status_widget.value = f"カメラは割り当てられていません"

camera_device_indices = [
    int(x[len(CAMERA_DEVICE_PREFIX):])
    for x in sorted(glob.glob(f"{CAMERA_DEVICE_PREFIX}*"))
]
camera_devices_widget.options = camera_device_indices
camera_devices_widget.observe(start_camera, names="value")
if len(camera_device_indices) > 0:
    camera_devices_widget.value = camera_device_indices[0]
snapshot_button.on_click(take_snapshot)
save_button.on_click(save_snapshot)
stop_camera_button.on_click(stop_camera)

In [ ]:
%%capture
import atexit

# カーネル終了時にカメラが停止されるようにする
atexit.register(stop_camera)

In [ ]:
# UIを表示
display(ipywidgets.VBox([
    camera_devices_widget,
    ipywidgets.HBox([camera_widget, snapshot_widget]),
    draw_grids_checkbox,
    ipywidgets.HBox([snapshot_button, save_button]),
    filename_widget,
    stop_camera_button,
    status_widget
]))